# Plotting function used with the functional name dataset

In [ ]:
import warnings

import pandas as pd
from h5py.h5z import FLAG_SKIP_EDC
import numpy as np
import math
import matplotlib.pyplot as plt

from pandas.io.sas.sas_constants import sas_date_formats

# from Scaling_data import column_names
from utils import *


In [ ]:
# df_1_hME1_scaled = pd.read_excel("Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1_functional_names_diff_scaled_phPlus.xlsx")
# df_1_hME1_scaled = df_1_hME1_scaled.fillna(0)

df_2_hME1_scaled = pd.read_excel("Experiment/2_hTERT_HME1/Data/Processed/Full_dataset_2_hTERT_HME1_functional_names_diff_scaled_phPlus_clustered.xlsx")
df_2_hME1_scaled = df_2_hME1_scaled.fillna(0)

df_1_HEK_scaled = pd.read_excel("Experiment/1_HEK293T/Data/Processed/Full_dataset_HEK293T_functional_names_diff_scaled_phPlus.xlsx")
df_1_HEK_scaled = df_1_HEK_scaled.fillna(0)

In [ ]:
filtered_df = df_2_hME1_scaled.copy()

data_type = "log2_FC"
exclude_full = False
condition_for_clustering = []
column_names = filtered_df.columns.tolist()
if len(condition_for_clustering) == 0:
    column_selection = [element for element in column_names if element.startswith(f"{data_type}")] #f"{data_type}" in element]
    if exclude_full == True:
        column_selection = [element for element in column_names if element.startswith(f"{data_type}") and "full" not in element]
else:
    column_selection = [element for element in column_names if element.startswith(f"{data_type}") and any(cond in element for cond in condition_for_clustering)]
    if exclude_full == True:
        column_selection = [element for element in column_names if element.startswith(f"{data_type}") and any(cond in element for cond in condition_for_clustering) and "full" not in element]


mask = filtered_df[column_selection].abs().max(axis=1) > 0.5
filtered_df = filtered_df.loc[mask].copy()
print(f"len(subdata_fc) [FC >0.5] = {len(filtered_df)}")

In [ ]:
proteins_sites = {"IRS1" : ["Y632", "Y941"],
                  "EGFR" : ["T693"]
                  }

# subset_proteins = df_2_hME1_scaled.loc[(df_2_hME1_scaled["protein_name"].isin(proteins_sites.keys()))].copy()
# subset_sites = subset_proteins.loc[subset_proteinsc["site"].isin(proteins_sites.values())].copy()
# Build a regex per protein: \b(Y632|Y941)\b etc., escaping any special characters

# subset_proteins


In [ ]:
proteins= ["EGFR"] #TSC2

plot_protein_phosphosites(df_2_hME1_scaled, data_type = "log2",
                          proteins = proteins, replicates = False,  exclude_rep = [], #exclude_rep in format: 'rep1', 'rep2', 'rep3', 'rep4'
                          legend_plot = ['1.58 nM EGF', '100 nM INS', '1.58 nM EGF + 10 nM INS'], color_palette = ['r', 'b', 'fuchsia'], #['r', 'b', 'fuchsia']
                          saving_path = "",
                          saving_info = "", title_info = "",
                          plot_individually = False, fit_y_lims = False,
                          plot_close = False,
                          save_pdf = False, save_png = False,)

# uniprot_links_for(df_2_hME1_scaled,proteins)


In [ ]:

fig = plot_volcano_interactive_plotly_highlighting(
    df_2_hME1_scaled,
    fc_col="log2_FC_EGFnINS_5",
    pval_col="FC_pvalue_EGFnINS_5",
    site_col="site",  # or "found_phosphosites"
    highlight_proteins=["EGFR","SHC","SRC","GRB2","SOS1","GAB1","AKT1","PLCB3","IRS1"],  # mix of protein_Id/protein_name works
    match_cols=("protein_Id", "protein_name"),
    case_insensitive=True,
    show_highlight_labels=False,
    title=""
)
fig.show()

In [ ]:
############## ["BRAF", "q15418" -ribosomal RPS6KA1, GAB1, P29590
### ["EGFR", "SOS1", "BRAF", "MAP2K1", 'KSR1', 'SHOC2', 'GAB1', 'PAK1', 'STAM', 'STAM2', 'RPS6KA3', 'AHNAK', 'AHNAK2']
protein_list = ["EGFR","SOS1","BRAF", "GAB1", "KRT13", "RPS6KA1"]
############ Single plot
# plot_volcano(df_2_hME1_scaled, "log2_FC_EGF_2", "FC_pvalue_EGF_2",
#              fc_thresh=1.0,
#              pval_thresh=0.05,
#              # title="log2_FC_EGF_10",
#              highlight_proteins=protein_list,
#              fit_x_limit = False)

############ In a 3×7 grid:
column_names = df_2_hME1_scaled.columns.to_list()
data_type1 = "log2_FC"
data_type2 = "FC_pvalue"
log2_FCs = [element for element in column_names if data_type1 in element]
FC_pvalues = [element for element in column_names if data_type2 in element]
pair_list = list(zip(log2_FCs, FC_pvalues))

# For only one subset if wanted (change the subplot size
# new_pair_list = pair_list[:7]
# # print(new_pair_list)
#
fig, axes = plt.subplots(3, 7, figsize=(30, 15)) #figsize=(18, 10)
axes = axes.flatten()
for i, (fc_col, pval_col) in enumerate(pair_list):  # pairs built as earlier
    plot_volcano(df_2_hME1_scaled, fc_col, pval_col,
                 highlight_proteins=protein_list,
                 ax=axes[i], title=fc_col,
                 fit_x_limit=True)
plt.tight_layout()
plt.show()

In [ ]:
erk_site = df_2_hME1_scaled[["protein_name", "protein_Id", "site", "ERK_motif", "functional_score"]].copy()

erk_site.loc[erk_site["protein_name"] == "ARAF"]

In [ ]:
df_2_hME1_scaled

In [ ]:
df_2_hME1_scaled.loc[df_2_hME1_scaled["protein_Id"] == "Q9NYT0"]

In [ ]:
fig = px.scatter(
        df_2_hME1_scaled,
        x=df_2_hME1_scaled.log2_FC_EGF_15,
        y=df_2_hME1_scaled.log2_FC_EGFnINS_15,
        hover_name=df_2_hME1_scaled["site"],
        # hover_data={fc_col: ':.3f', "neglog10p": ':.3f', pval_col: ':.3g'},
        # title=title or f"Volcano: {fc_col}",
        template="plotly_white"
)

# Threshold lines
# fig.add_hline(y=-np.log10(pval_thresh), line_dash="dash")
# fig.add_vline(x=fc_thresh, line_dash="dash")
# fig.add_vline(x=-fc_thresh, line_dash="dash")

fig.add_shape(
    type="line",
    x0=-5, y0=-5,
    x1=5, y1=5,  # normalized by default (if not using xref/yref)
    xref="x", yref="y",  # important: make it use data coordinates
    line=dict(color="red", dash="dash"),
)

fig.update_layout(
    xaxis_title="Log2_FC EGF 5 mins",
    yaxis_title="Log2_FC EGFnINS 5 mins",
    legend_title="",
    height=600,
    width=800,
)

In [ ]:
plot_volcano(df_2_hME1_scaled, fc_col, pval_col,
                 highlight_proteins=protein_list,
                 ax=axes[i], title=fc_col,
                 fit_x_limit=True)